# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the [FAIR^2 Mass Spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)

# Access the metadata object (don't subscript or iterate)
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version} | License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished} | Identifier: {dataset.metadata.identifier}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# The mlcroissant metadata exposes record sets as a list of RecordSet objects
recordsets = getattr(dataset.metadata, 'recordSet', [])
print(f"Number of Record Sets: {len(recordsets)}\n")

for i, rs in enumerate(recordsets):
    print(f"[{i}] Record Set @id: {rs['@id']} | name: {rs.get('name', '(unnamed)')}")
    fields = rs.get('field', [])
    print(f"       Fields (by @id):")
    for f in fields:
        if isinstance(f, dict):
            print(f"         - {f.get('@id')} (name: {f.get('name', '')})")
        else:
            print(f"         - {f}")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in getattr(dataset.metadata, 'recordSet', [])]
print(f"Record set @ids: {record_set_ids}\n")

# Load records for each record set into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print(f"  (No records found or not a tabular record set.)\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming distributions, or grouping by key attributes.

In [ ]:
# For EDA, pick the first tabular record set if available
record_set_id = None
for k, df in dataframes.items():
    if not df.empty:
        record_set_id = k
        break
assert record_set_id is not None, "No tabular record sets loaded."
df = dataframes[record_set_id]
print(f"Using record set {record_set_id} for EDA.")

# Choose a numeric field (try common proteomics fields such as 'MW', 'Abundance', etc.)
# If columns have units in the name (e.g. MW), adjust accordingly.
numeric_col_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if not numeric_col_candidates:
    # Otherwise, detect float columns by attempting conversion
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
            numeric_col_candidates.append(c)
        except:
            continue
print(f"Numeric field candidates: {numeric_col_candidates}")

if numeric_col_candidates:
    numeric_field = numeric_col_candidates[0]
    threshold = df[numeric_field].mean() if pd.notna(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a common field, e.g. 'Description', or first non-numeric field
    group_field_candidates = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the chosen numeric field
if numeric_col_candidates:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in Record Set: {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a group_field exists, visualize mean values per group (top 20)
    if group_field:
        top_groups = grouped_df.sort_values(numeric_field, ascending=False).head(20)
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field, y=numeric_field, data=top_groups)
        plt.xticks(rotation=90)
        plt.title(f"Top 20 {group_field}s by mean {numeric_field}")
        plt.show()
else:
    print("No numeric data for visualization.")

## 6. Conclusion
In this notebook, we ingested and explored the [FAIR^2 Mass Spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using `mlcroissant`. We demonstrated how to programmatically examine dataset metadata, inspect record sets and fields by their `@id`, extract tabular data for analysis, and perform basic EDA and visualizations. This workflow can be extended for more advanced analysis such as protein biomarker discovery, cross-condition comparison, or integration with external proteomics resources.